# MediCare Patient Follow-Up Agent

**Domain:** Healthcare | **Role:** Data Scientist | **Format:** Jupyter Notebook + CSV

---

## Scenario
MediCare Clinic manages over 1,000 chronic disease patients across conditions like Type 2 Diabetes, Hypertension, Heart Failure, COPD, and Anemia. The clinical team needs an AI-powered agent that can:
- Autonomously review patient records
- Identify patients at clinical risk
- Generate prioritised follow-up care action plans

This notebook implements a **prototype Agentic AI system** using the **Groq LLM API** (`llama-3.1-8b-instant`) with native tool-calling to orchestrate clinical decision-making.

### Notebook Structure
| Task | Description |
|------|-------------|
| Task 1 | Data Loading & Exploratory Analysis |
| Task 2 | Define Agent Tools |
| Task 3 | Implement the Agentic Loop |
| Task 4 | Single Patient Analysis |
| Task 5 | Missed Appointment Follow-Up |

In [ ]:
# Install required packages if not already installed
# !pip install pandas groq matplotlib seaborn -q

import pandas as pd
import json
from groq import Groq

# Groq API Configuration
# llama-3.1-8b-instant is a great free model on Groq with native tool-calling support
# Get a free API key at: https://console.groq.com
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"  # <-- Replace with your actual Groq API key
MODEL        = "llama-3.1-8b-instant"

client = Groq(api_key=GROQ_API_KEY)
print(f"Groq client initialised | Model: {MODEL}")

---

## Task 1 - Data Loading & Exploratory Analysis

Load `patient_data.csv` and perform EDA to understand dataset structure, patient demographics, clinical distributions, and appointment adherence.

In [ ]:
# Load patient dataset
df = pd.read_csv("patient_data.csv", parse_dates=["last_visit_date", "next_scheduled_visit"])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Exploratory Data Analysis & Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 55 + "\n DATASET OVERVIEW\n" + "=" * 55)
df['primary_diagnosis'] = df['diagnosis'].apply(lambda x: str(x).split(',')[0].strip())

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df['age'], bins=20, ax=axes[0], color='skyblue').set_title('Age Distribution')
sns.countplot(data=df, y='primary_diagnosis', ax=axes[1], order=df['primary_diagnosis'].value_counts().index, palette='Set2').set_title('Primary Diagnoses')
sns.countplot(data=df, x='missed_last_appointment', ax=axes[2], palette='pastel').set_title('Missed Appointments')
plt.tight_layout()
plt.show()

# Print concise summary
print(f"Total Patients: {len(df)} | Missing values: {df.isnull().sum()[df.isnull().sum() > 0].to_dict() or 'None'}")
print(f"Age: mean={df['age'].mean():.1f} | Missed Appt -> Yes: {sum(df['missed_last_appointment']=='Yes')}")
print(f"Vitals: BP(Sys) mean={df['vitals_bp_systolic'].mean():.1f}, HR mean={df['vitals_heart_rate'].mean():.1f}")

In [ ]:
# Clinical Risk Thresholds (evidence-based, per medical guidelines)
# These structured thresholds power the flag_clinical_risk tool.
RISK_THRESHOLDS = {"HbA1c": {"critical": 10.0, "high": 8.0, "unit": "%", "direction": "above"}, "BNP": {"critical": 900, "high": 400, "unit": "pg/mL", "direction": "above"}, "eGFR": {"critical": 30, "high": 45, "unit": "mL/min", "direction": "below"}, "BP Systolic": {"critical": 180, "high": 140, "unit": "mmHg", "direction": "above"}, "FEV1%": {"critical": 30, "high": 50, "unit": "%", "direction": "below"}, "Hemoglobin": {"critical": 7.0, "high": 8.0, "unit": "g/dL", "direction": "below"}, "PHQ-9 Score": {"critical": 20, "high": 15, "unit": "/27", "direction": "above"}, "Peak Flow": {"critical": 200, "high": 320, "unit": "L/min", "direction": "below"}, "Pain Score": {"critical": 8, "high": 6, "unit": "/10", "direction": "above"}}
print(f"Risk thresholds configured for {len(RISK_THRESHOLDS)} lab test types")

---

## Task 2 - Define Agent Tools

Define **4 tools** the LLM agent can autonomously invoke:
1. `get_patient_record` - Retrieve full patient record by ID
2. `flag_clinical_risk` - Assess risk level (LOW / HIGH / CRITICAL)
3. `generate_followup_action` - Create prioritised care action plan
4. `get_missed_appointment_patients` - List all missed-appointment patients

Each tool has:
- A **Python implementation** that queries the live DataFrame
- A **JSON schema** that the LLM uses to understand how and when to call the tool

In [ ]:
# Tool 1: get_patient_record
def get_patient_record(patient_id: str) -> dict:
    """Retrieve all clinical data for a patient by their ID."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return {"error": f"Patient {patient_id} not found."}
    record = row.iloc[0].to_dict()
    for k, v in record.items():
        if hasattr(v, "strftime"):
            record[k] = v.strftime("%Y-%m-%d")
    return record

# Tool 2: flag_clinical_risk
def flag_clinical_risk(patient_id: str) -> dict:
    """Assess clinical risk (LOW/HIGH/CRITICAL) from lab values, vitals, and visit history."""
    record = get_patient_record(patient_id)
    if "error" in record:
        return record
    flags = []
    lab_test  = record.get("lab_test", "")
    lab_value = float(record.get("lab_value", 0))
    thresh    = RISK_THRESHOLDS.get(lab_test, {})

    if thresh:
        d = thresh["direction"]
        if d == "above":
            if lab_value >= thresh["critical"]:
                flags.append(f"CRITICAL: {lab_test}={lab_value}{record['lab_unit']} (>={thresh['critical']})")
            elif lab_value >= thresh["high"]:
                flags.append(f"HIGH: {lab_test}={lab_value}{record['lab_unit']} (>={thresh['high']})")
        else:
            if lab_value <= thresh["critical"]:
                flags.append(f"CRITICAL: {lab_test}={lab_value}{record['lab_unit']} (<={thresh['critical']})")
            elif lab_value <= thresh["high"]:
                flags.append(f"HIGH: {lab_test}={lab_value}{record['lab_unit']} (<={thresh['high']})")

    if record.get("missed_last_appointment") == "Yes":
        flags.append("MISSED: Patient missed their last scheduled appointment")

    if record.get("days_since_last_visit", 0) > 180:
        flags.append(f"OVERDUE: Last visit was {record['days_since_last_visit']} days ago")

    if record.get("vitals_bp_systolic", 0) >= 180:
        flags.append(f"CRITICAL: BP Systolic={record['vitals_bp_systolic']} mmHg - hypertensive crisis risk")

    if record.get("vitals_spo2", 100) < 92:
        flags.append(f"HIGH: SpO2={record['vitals_spo2']}% - below safe threshold")

    risk_level = ("CRITICAL" if any("CRITICAL" in f for f in flags)
                  else "HIGH" if flags else "LOW")
    return {"patient_id": patient_id, "patient_name": record["patient_name"],
            "diagnosis": record["diagnosis"], "risk_level": risk_level,
            "flags": flags or ["No critical flags detected"]}

# Tool 3: generate_followup_action
def generate_followup_action(patient_id: str, risk_level: str, flags: list) -> dict:
    """Generate a structured follow-up care action plan based on risk level and clinical flags."""
    record = get_patient_record(patient_id)
    if "error" in record:
        return record
    plan_map = {"CRITICAL": {"urgency": "Immediate (24 hours)", "action": "Emergency outreach + urgent clinic appointment", "follow_up": "1-3 days"}, "HIGH": {"urgency": "High priority (3-5 days)", "action": "Phone consultation + expedited appointment", "follow_up": "1-2 weeks"}, "LOW": {"urgency": "Routine", "action": "Standard monitoring at next scheduled visit", "follow_up": "As scheduled"}}
    p = plan_map.get(risk_level, plan_map["LOW"])
    return {"patient_id": patient_id, "patient_name": record["patient_name"],
            "age": record["age"], "diagnosis": record["diagnosis"],
            "current_medication": record["current_medication"],
            "risk_level": risk_level, "clinical_flags": flags,
            "urgency": p["urgency"], "recommended_action": p["action"],
            "follow_up_in": p["follow_up"], "clinical_notes": record.get("notes", "")}

# Tool 4: get_missed_appointment_patients
def get_missed_appointment_patients() -> dict:
    """Retrieve all patients who missed their last scheduled appointment."""
    cols = ["patient_id","patient_name","age","diagnosis","days_since_last_visit","next_scheduled_visit"]
    missed = df[df["missed_last_appointment"] == "Yes"][cols].copy()
    missed["next_scheduled_visit"] = missed["next_scheduled_visit"].astype(str)
    return {"count": len(missed), "patients": missed.to_dict(orient="records")}

print("4 tools implemented: get_patient_record, flag_clinical_risk, generate_followup_action, get_missed_appointment_patients")

In [ ]:
# JSON Schemas for Groq API tool-calling
# These schemas tell the LLM what tools exist, their purpose, and expected parameters.
TOOLS = [
    {"type": "function", "function": {
        "name": "get_patient_record",
        "description": "Retrieve complete clinical data for a patient using their patient_id.",
        "parameters": {"type": "object",
            "properties": {"patient_id": {"type": "string", "description": "Patient ID e.g. P0001"}},
            "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "flag_clinical_risk",
        "description": "Assess clinical risk level (LOW/HIGH/CRITICAL) from lab values, vitals, and visit history.",
        "parameters": {"type": "object",
            "properties": {"patient_id": {"type": "string"}},
            "required": ["patient_id"]}}},
    {"type": "function", "function": {
        "name": "generate_followup_action",
        "description": "Generate a structured follow-up care action plan given patient ID, risk level, and clinical flags.",
        "parameters": {"type": "object",
            "properties": {
                "patient_id" : {"type": "string"},
                "risk_level" : {"type": "string", "enum": ["LOW","HIGH","CRITICAL"]},
                "flags"      : {"type": "array", "items": {"type": "string"}}},
            "required": ["patient_id","risk_level","flags"]}}},
    {"type": "function", "function": {
        "name": "get_missed_appointment_patients",
        "description": "Retrieve all patients who missed their last appointment for outreach prioritisation.",
        "parameters": {"type": "object", "properties": {}, "required": []}}}
]

# Dispatch map: maps tool name strings to their Python implementations
TOOL_DISPATCH = {
    "get_patient_record"              : get_patient_record,
    "flag_clinical_risk"              : flag_clinical_risk,
    "generate_followup_action"        : generate_followup_action,
    "get_missed_appointment_patients" : get_missed_appointment_patients
}
print("Tool schemas and dispatch map ready")

---

## Task 3 - Implement the Agentic Loop

Implements a **ReAct-style** agentic loop (Reason -> Act -> Observe -> Repeat):

```
User Query
    |
    v
[LLM] -- decides tool to call
    |
    v
[Tool] -- executes, returns result
    |
    v
[LLM] -- reasons over result, decides next tool OR final answer
    |
    v
Final Response
```

The LLM autonomously chains multiple tool calls before arriving at a recommendation, mimicking real clinical reasoning.

In [ ]:
# System Prompt - defines the agent's role and behaviour
SYSTEM_PROMPT = """You are MediCare AI, a clinical decision-support agent for MediCare Clinic.

Your responsibilities:
- Analyse patient records using the available tools
- Identify patients at HIGH or CRITICAL clinical risk
- Generate clear, actionable follow-up care plans for the care coordination team
- Always use tools to retrieve real data -- do NOT invent clinical information
- Be structured and clinically precise in your responses

Risk guidance:
- CRITICAL: Act within 24 hours (emergency outreach)
- HIGH: Act within 3-5 days (priority consultation)
- LOW: Standard monitoring at next scheduled visit"""

def run_agent(user_query: str, verbose: bool = True) -> str:
    """
    Execute the ReAct-style agentic loop for a given clinical query.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_query}
    ]

    for iteration in range(10):  # Safety cap: max 10 LLM round-trips per query
        # Step 1: Call LLM with available tools
        response   = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            tool_choice="auto", temperature=0.1, max_tokens=2048
        )
        msg        = response.choices[0].message
        tool_calls = msg.tool_calls

        # Append assistant message to conversation history
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                             "function": {"name": tc.function.name,
                                          "arguments": tc.function.arguments}}
                            for tc in tool_calls] if tool_calls else []
        })

        # Step 2: No tool calls = LLM has reached its final answer
        if not tool_calls:
            return msg.content

        # Step 3: Execute each tool call and feed result back to LLM
        for tc in tool_calls:
            fn_name   = tc.function.name
            fn_args   = json.loads(tc.function.arguments)
            if verbose:
                print(f"  [Tool call {iteration+1}] {fn_name}({fn_args})")
            fn_result = TOOL_DISPATCH[fn_name](**fn_args)   # Dispatch to Python function
            if verbose:
                print(f"  [Result]    {str(fn_result)[:250]}\n")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                              "content": json.dumps(fn_result)})

    return "Agent reached maximum iterations without a final answer."

print("Agentic loop defined - ready to run queries")

---

## Task 4 - Single Patient Analysis

Run the agent on individual patients. The agent autonomously:
1. Retrieves the patient's full clinical record
2. Flags risk level with supporting evidence (lab values, vitals, visit history)
3. Generates a tailored, prioritised follow-up action plan

Two examples are shown: one diabetic/CKD patient (P0003) and one critical cardiac patient (P0008).

In [ ]:
# Single Patient Analysis: P0003 - Rohan Mehta, 45M, Type 2 Diabetes + CKD, HbA1c=11.4% (critical)
print("=" * 60)
print(" SINGLE PATIENT ANALYSIS -- P0003")
print("=" * 60 + "\n")

result_p0003 = run_agent(
    "Perform a complete clinical review for patient P0003: "
    "retrieve their record, assess clinical risk, and generate a prioritised follow-up action plan.",
    verbose=True
)

print("\n" + "-" * 60)
print("AGENT RESPONSE -- P0003")
print("-" * 60)
print(result_p0003)

In [ ]:
# Single Patient Analysis: P0008 - Deepa Krishnan, Heart Failure, BNP=1100 pg/mL (critical)
print("=" * 60)
print(" SINGLE PATIENT ANALYSIS -- P0008")
print("=" * 60 + "\n")

result_p0008 = run_agent(
    "Perform a clinical risk assessment for patient P0008 who has Heart Failure. "
    "Retrieve their record, flag clinical risk, and generate a detailed follow-up action plan.",
    verbose=True
)

print("\n" + "-" * 60)
print("AGENT RESPONSE -- P0008")
print("-" * 60)
print(result_p0008)

---

## Task 5 - Missed Appointment Follow-Up

The agent identifies **all patients who missed their last appointment** and produces a prioritised outreach plan ranked by clinical urgency (CRITICAL > HIGH > LOW). This enables the care coordination team to act on the highest-risk patients first.

In [ ]:
# Missed Appointment Bulk Follow-Up - Agent-driven outreach prioritisation
print("=" * 60)
print(" MISSED APPOINTMENT FOLLOW-UP CAMPAIGN")
print("=" * 60 + "\n")

result_missed = run_agent(
    "Identify all patients who missed their last appointment. "
    "For each patient, retrieve their record and flag clinical risk. "
    "Rank patients by urgency (CRITICAL > HIGH > LOW) and provide a structured "
    "outreach action plan the care coordination team can act on today.",
    verbose=True
)

print("\n" + "-" * 60)
print("MISSED APPOINTMENT OUTREACH PLAN")
print("-" * 60)
print(result_missed)

In [ ]:
# Summary statistics for missed-appointment patients (programmatic validation)
missed_df = df[df['missed_last_appointment'] == 'Yes'].copy()
print(f"Total patients who missed appointment: {len(missed_df)}")
print(f"\nBreakdown by primary diagnosis:")
print(missed_df['primary_diagnosis'].value_counts().to_string())
print(f"\nAvg days since last visit: {missed_df['days_since_last_visit'].mean():.1f}")
print(f"Max days since last visit: {missed_df['days_since_last_visit'].max()}")

# Display risk flags for each missed-appointment patient using our tool
print("\nRisk Assessment for Missed-Appointment Patients:")
print(f"{'Patient ID':<12} {'Name':<22} {'Risk':<10} {'Top Clinical Flag'}")
print("-" * 90)
for _, row in missed_df.iterrows():
    risk = flag_clinical_risk(row['patient_id'])
    print(f"{row['patient_id']:<12} {row['patient_name']:<22} {risk['risk_level']:<10} {risk['flags'][0]}")

---

## Summary

### What Was Built

| Component | Detail |
|-----------|--------|
| **LLM** | Groq `llama-3.1-8b-instant` - fast, free model with native tool-calling |
| **Tools** | 4 tools: record retrieval, risk flagging, action plan generation, missed appt lookup |
| **Agent Loop** | ReAct-style: LLM reasons -> calls tools -> gets results -> reasons again |
| **Risk Engine** | Evidence-based thresholds for 9 lab types + vitals + visit history |
| **Output** | Prioritised, structured follow-up action plans per patient |

### Clinical Risk Thresholds Applied

| Condition | Lab Test | High Threshold | Critical Threshold |
|-----------|----------|---------------|--------------------|
| Type 2 Diabetes | HbA1c | >=8% | >=10% |
| Heart Failure | BNP | >=400 pg/mL | >=900 pg/mL |
| CKD | eGFR | <=45 mL/min | <=30 mL/min |
| Hypertension | BP Systolic | >=140 mmHg | >=180 mmHg |
| COPD | FEV1% | <=50% | <=30% |
| Anemia | Hemoglobin | <=8 g/dL | <=7 g/dL |
| Depression | PHQ-9 | >=15/27 | >=20/27 |
| Asthma | Peak Flow | <=320 L/min | <=200 L/min |
| Osteoarthritis | Pain Score | >=6/10 | >=8/10 |

### Key Design Decisions
- **No hardcoded action logic**: The LLM decides which actions to take; only data retrieval and risk categorisation are deterministic
- **Tool-first architecture**: All clinical data flows through well-defined tool interfaces, making the system auditable and extensible
- **ReAct loop**: The agent chains multiple tool calls before arriving at a final recommendation, mimicking clinical reasoning
- **Safety cap**: Maximum 10 LLM iterations per query prevents infinite loops
- **Low temperature (0.1)**: Ensures consistent, reproducible clinical recommendations across runs